# Beschreibung: 

# Importe:

In [1]:
import sys
import os

import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules, compute_coverage

os.getcwd()

'/home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen'

# Daten laden:

Student Performance & Behavior Dataset: https://www.kaggle.com/datasets/mahmoudelhemaly/students-grading-dataset?select=Students_Grading_Dataset_Biased.csv

In [2]:
biased = pd.read_csv("../Daten/Student_Performance_Behavior_Dataset/Students_Grading_Dataset_Biased.csv")
unbiased = pd.read_csv("../Daten/Student_Performance_Behavior_Dataset/Students_Performance_Dataset.csv")

print(biased.shape)
print(unbiased.shape)

(5000, 23)
(5000, 23)


# Ausführung

### Vorbereitung: (Datenaufbereitung)

In [3]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Grade"

In [4]:
cutoffs = {
    "age": [20, 22], # Aufteilung in 18/19, 20/21, 22/23/24
    "Sleep_Hours_per_Night": [6, 8], # Aufteilung in 4/5, 6/7, 8/9  -  wenig, Durchschnitt, viel
    "Total_Score":[60, 70, 80, 90]  # Wichtig: Übliche Vergabe von Noten F: 0-60), D: 60-70), C: 70-80), B: 80-90), A: 90-100)
}

# Diskretisierung der numerischen Daten:
biased_disc = discretize(biased, bins=4, cutoffs=cutoffs)
unbiased_disc = discretize(unbiased, bins=4,cutoffs=cutoffs)

# „Pass“ wieder anfügen, damit es NICHT diskretisiert wird.
biased_disc["Pass"] = (biased["Grade"].isin(["A", "B", "C", "D"])).astype(int)
unbiased_disc["Pass"] = (unbiased["Grade"].isin(["A", "B", "C", "D"])).astype(int)

In [5]:
# Konditionsattribute (Alle Werte - bis auf Student_ID und Email: conditional attributs all)
cond_attrs = []
for col in biased_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte (z. B. Total_Score_disc)
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte (Gender, Department etc.)
        elif biased_disc[col].dtype == "object":
            cond_attrs.append(col)

cond_attrs.remove('Student_ID') # Das ist ein Identifier, daher muss es raus.
cond_attrs.remove('Email') # Es gilt hier dasselbe
cond_attrs.remove('First_Name') # Das ist zwar kein Identifier, sorgt aber für einen sehr starke Streuung an Äquivalenzklassen. Im Beipsieldatensatz zwar nicht und kann daher da auch verwendet werden, aber sonst müsste daas raus.
cond_attrs.remove('Last_Name') # Es gilt hier dasselbe

print(cond_attrs)

['Gender', 'Department', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Parent_Education_Level', 'Family_Income_Level', 'Age_disc', 'Attendance (%)_disc', 'Midterm_Score_disc', 'Final_Score_disc', 'Assignments_Avg_disc', 'Quizzes_Avg_disc', 'Participation_Score_disc', 'Projects_Score_disc', 'Total_Score_disc', 'Study_Hours_per_Week_disc', 'Stress_Level (1-10)_disc', 'Sleep_Hours_per_Night_disc']


### Datenbetrachtung:

#### cond_attrs: Datensatz mit nicht nur diskretisierten numerischen Werten sondern auch direkt kategorische Werte (Gender, Department etc.)

In [6]:
# Redukte
biased_reduct, info_biased_grade = quick_reduct(biased_disc, cond_attrs, decision_attr)
unbiased_reduct, info_unbiased_grade = quick_reduct(unbiased_disc, cond_attrs, decision_attr)

print("Biased Reduct:\n", biased_reduct)
print("\nUnbiased Reduct:\n", unbiased_reduct)

γ(C) mit allen Attributen: 0.516200
Einzel-γ-Werte:
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family_Income_Level: γ = 0.000000
  Age_disc: γ = 0.000000
  Attendance (%)_disc: γ = 0.000000
  Midterm_Score_disc: γ = 0.000000
  Final_Score_disc: γ = 0.000000
  Assignments_Avg_disc: γ = 0.000000
  Quizzes_Avg_disc: γ = 0.000000
  Participation_Score_disc: γ = 0.000000
  Projects_Score_disc: γ = 0.000000
  Total_Score_disc: γ = 0.000000
  Study_Hours_per_Week_disc: γ = 0.000000
  Stress_Level (1-10)_disc: γ = 0.000000
  Sleep_Hours_per_Night_disc: γ = 0.000000

Alle γ({a}) = 0, aber γ(C) > 0 → benutze quick_reduct_interaction.
γ(C) mit allen Attributen: 0.795000
Einzel-γ-Werte:
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family_Inco

In [7]:
rules_grade_unbiased = induce_rules(unbiased_disc, unbiased_reduct, decision_attr, verbose=True)
rules_grade_biased = induce_rules(biased_disc, biased_reduct, decision_attr, verbose=True)

print("unbaised:")
for r in rules_grade_unbiased[:10]:
    print(r)

print("\nbaised:")
for r in rules_grade_biased[:10]:
    print(r)

Anzahl an Klassen: 5

Anzahl an Klassen: 4004

unbaised:
{'premise': {'Total_Score_disc': np.int64(0)}, 'decision': 'F', 'support': 279}
{'premise': {'Total_Score_disc': np.int64(3)}, 'decision': 'B', 'support': 638}
{'premise': {'Total_Score_disc': np.int64(1)}, 'decision': 'D', 'support': 1760}
{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': 'C', 'support': 2307}
{'premise': {'Total_Score_disc': np.int64(4)}, 'decision': 'A', 'support': 16}

baised:
{'premise': {'Gender': 'Female', 'Department': 'Engineering', 'Extracurricular_Activities': 'No', 'Internet_Access_at_Home': 'Yes', 'Attendance (%)_disc': np.float64(0.0), 'Total_Score_disc': np.int64(0), 'Projects_Score_disc': np.int64(2), 'Quizzes_Avg_disc': np.int64(1), 'Study_Hours_per_Week_disc': np.int64(0)}, 'decision': 'F', 'support': 1}
{'premise': {'Gender': 'Female', 'Department': 'Engineering', 'Extracurricular_Activities': 'No', 'Internet_Access_at_Home': 'Yes', 'Attendance (%)_disc': np.float64(0.0), 'Total_Score_

In [8]:
biased_pass_reduct, info_biased_pass = quick_reduct(biased_disc, cond_attrs, "Pass")
unbiased_pass_reduct, info_unbiased_pass = quick_reduct(unbiased_disc, cond_attrs, "Pass")

print("Biased Reduct:\n", biased_pass_reduct)
print("\nUnbiased Reduct:\n", unbiased_pass_reduct)

γ(C) mit allen Attributen: 0.516200
Einzel-γ-Werte:
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family_Income_Level: γ = 0.000000
  Age_disc: γ = 0.000000
  Attendance (%)_disc: γ = 0.224200
  Midterm_Score_disc: γ = 0.000000
  Final_Score_disc: γ = 0.000000
  Assignments_Avg_disc: γ = 0.000000
  Quizzes_Avg_disc: γ = 0.000000
  Participation_Score_disc: γ = 0.000000
  Projects_Score_disc: γ = 0.000000
  Total_Score_disc: γ = 0.000000
  Study_Hours_per_Week_disc: γ = 0.000000
  Stress_Level (1-10)_disc: γ = 0.000000
  Sleep_Hours_per_Night_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.
γ(C) mit allen Attributen: 0.795000
Einzel-γ-Werte:
  Gender: γ = 0.000000
  Department: γ = 0.000000
  Extracurricular_Activities: γ = 0.000000
  Internet_Access_at_Home: γ = 0.000000
  Parent_Education_Level: γ = 0.000000
  Family

In [9]:
rules_pass_biased = induce_rules(biased_disc, biased_pass_reduct, "Pass", verbose=True)
rules_pass_unbiased = induce_rules(unbiased_disc, unbiased_pass_reduct, "Pass", verbose=True)

print("baised:")
for r in rules_pass_biased[:10]:
    print(r)

print("\nunbaised:")
for r in rules_pass_unbiased[:10]:
    print(r)

Anzahl an Klassen: 4

Anzahl an Klassen: 5

baised:
{'premise': {'Attendance (%)_disc': np.float64(3.0)}, 'decision': np.int64(1), 'support': 1121}

unbaised:
{'premise': {'Total_Score_disc': np.int64(0)}, 'decision': np.int64(0), 'support': 279}
{'premise': {'Total_Score_disc': np.int64(3)}, 'decision': np.int64(1), 'support': 638}
{'premise': {'Total_Score_disc': np.int64(1)}, 'decision': np.int64(1), 'support': 1760}
{'premise': {'Total_Score_disc': np.int64(2)}, 'decision': np.int64(1), 'support': 2307}
{'premise': {'Total_Score_disc': np.int64(4)}, 'decision': np.int64(1), 'support': 16}


# Resultate

In [10]:
print("Ergebnisse:")

print("\nBiased:")
print("Pass:", biased_pass_reduct)
print("Pass Rules:", len(rules_pass_biased))
print("Grade:", biased_reduct) # A, B
print("Grade Rules:", len(rules_grade_biased))
#print("Pass Rules:", rules_pass_biased[2]) # Einzeln
#print("Pass Rules:", rules_pass_biased) # Das wären alle

print("\nUnbiased:")
print("Pass:", unbiased_pass_reduct)
print("Pass Rules:", len(rules_pass_unbiased))
print("Grade:", unbiased_reduct) # A, B
print("Grade Rules:", len(rules_grade_unbiased))

Ergebnisse:

Biased:
Pass: ['Attendance (%)_disc']
Pass Rules: 1
Grade: ['Gender', 'Department', 'Extracurricular_Activities', 'Internet_Access_at_Home', 'Attendance (%)_disc', 'Total_Score_disc', 'Projects_Score_disc', 'Quizzes_Avg_disc', 'Study_Hours_per_Week_disc']
Grade Rules: 3707

Unbiased:
Pass: ['Total_Score_disc']
Pass Rules: 5
Grade: ['Total_Score_disc']
Grade Rules: 5


In [11]:
#ind = indiscernibility(biased_disc, biased_pass_reduct)
#ind

# Wiedergabe der Abdeckung durch Regeln:
### Hier nur zum Bestehen

In [12]:
# Annahme: Folgendes muss vorhanden sein:
# biased_disc, unbiased_disc
# biased_pass_reduct, unbiased_pass_reduct
# rules_pass_biased, rules_pass_unbiased

cov_biased = compute_coverage(
    biased_disc,
    biased_pass_reduct,
    "Pass",
    rules_pass_biased
)

cov_unbiased = compute_coverage(
    unbiased_disc,
    unbiased_pass_reduct,
    "Pass",
    rules_pass_unbiased
)

print("Coverage biased   :", cov_biased)
print("Coverage unbiased :", cov_unbiased)

cov_biased_pct   = cov_biased * 100
cov_unbiased_pct = cov_unbiased * 100

Coverage biased   : 0.2242
Coverage unbiased : 1.0


In [13]:
import matplotlib.pyplot as plt

labels = ["Biased", "Unbiased"]
values = [cov_biased_pct, cov_unbiased_pct]

x = np.arange(len(labels))

plt.figure()
plt.bar(x, values)
plt.xticks(x, labels)
plt.ylabel("Coverage [%]")
plt.title("Anteil deterministisch abgedeckter Studierender")

# optional: Werte über die Balken schreiben
for i, v in enumerate(values):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center", va="bottom")

plt.ylim(0, 100)  # da es Prozent sind
plt.tight_layout()
plt.savefig("coverage_biased_unbiased_3.pdf")
plt.close()


### Hier für Note:

In [14]:
cov_biased = compute_coverage(
    biased_disc,
    biased_reduct,
    "Grade",
    rules_grade_biased
)

cov_unbiased = compute_coverage(
    unbiased_disc,
    unbiased_reduct,
    "Graade",
    rules_grade_unbiased
)

print("Coverage grade biased   :", cov_biased)
print("Coverage grade unbiased :", cov_unbiased)

cov_biased_pct   = cov_biased * 100
cov_unbiased_pct = cov_unbiased * 100

Coverage grade biased   : 0.7698
Coverage grade unbiased : 1.0
